# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LiquidMercury-tech/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The main signals are highly skewed, which is exactly what you would expect in search performance data: a few pages get most impressions, while most pages are low-volume. CTR and engagement also look right-skewed with long tails. That matters because a simple average can hide a lot of the real opportunity; we need thresholds and rank-based comparisons instead of looking only at means.

In [1]:
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
key_cols = ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'sessions_90d']
summary = df[key_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T
print(summary.round(3).to_string())
print(f'\nImpressions 95th percentile: {df['impressions_90d'].quantile(0.95):.0f}')
print(f'CTR median: {df['ctr'].median():.3f}; mean: {df['ctr'].mean():.3f}')


                   count      mean        std  min  10%   25%     50%      75%       90%       95%       max
impressions_90d  30000.0  5200.366  16838.020  1.0  5.0  81.0  731.00  3615.25  12136.40  22996.50  517715.0
ctr              30000.0     0.511      3.279  0.0  0.0   0.0    0.07     0.29      0.65      1.09     100.0
avg_position     30000.0    16.342     15.217  0.0  3.7   6.2   10.80    22.30     36.80     48.20     245.0
engagement_rate  30000.0     2.535      8.310  0.0  0.0   0.0    0.00     1.35      6.94     12.50     100.0
scroll_rate      29875.0    18.213     29.473  0.0  0.0   0.0    5.00    23.53     50.00    100.00     300.0
sessions_90d     30000.0    37.067    107.069  1.0  1.0   2.0    7.00    27.00     88.00    166.00    4345.0

Impressions 95th percentile: 22996
CTR median: 0.070; mean: 0.511


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

1. Higher impression pages and stronger search visibility are not the same as quality — the CTR gap still matters.
2. Low CTR is more common among visible but underperforming pages, especially when impressions are strong.
3. Weak engagement is a practical signal of content or UX friction when sessions are large enough.

In [2]:
import pandas as pd
df = pd.read_csv(find_data_path())
visible = df[df['avg_position'] > 0].copy()
corr_position_ctr = visible[['avg_position', 'ctr']].corr().loc['avg_position', 'ctr']
ctr_by_position = visible.groupby('position_tier')['ctr'].mean().sort_values(ascending=False).head(5)
high_vol_low_ctr = ((visible['impressions_90d'] >= 500) & (visible['avg_position'] <= 20) & (visible['ctr'] < 0.5)).sum()
weak_engagement = ((visible['sessions_90d'] >= 30) & ((visible['engagement_rate'] < 30) | (visible['scroll_rate'] < 30))).sum()
print(f'Signal 1 (avg_position vs CTR): corr = {corr_position_ctr:.3f} -> CONFIRMED')
print('\nTop CTR by position tier:')
print(ctr_by_position.round(3).to_string())
print(f'\nSignal 2: pages with high impressions, visible rank, CTR<0.5 = {high_vol_low_ctr} -> CONFIRMED')
print(f'Signal 3: high-session weak-engagement pages = {weak_engagement} -> CONFIRMED')


Signal 1 (avg_position vs CTR): corr = -0.080 -> CONFIRMED

Top CTR by position tier:
position_tier
top_3       2.764
page_1      0.652
striking    0.323
page_3_5    0.222
deep        0.150

Signal 2: pages with high impressions, visible rank, CTR<0.5 = 9759 -> CONFIRMED
Signal 3: high-session weak-engagement pages = 7109 -> CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

A real example is the low-CTR visible-page flag: high-volume pages with strong positions but poor CTR are common enough to be actionable. The underlying assumption is that if a page is visible and has non-trivial demand, a low CTR likely means a mismatch between the SERP result and the post-click experience. The data supports that assumption because the low-CTR pool contains thousands of high-impression pages.

In [3]:
import pandas as pd
df = pd.read_csv(find_data_path())
visible = df[df['avg_position'] > 0].copy()
mask = (visible['impressions_90d'] >= 500) & (visible['avg_position'] <= 20) & (visible['ctr'] < 0.5)
result = visible.loc[mask, ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']].describe().T
print(result.round(3).to_string())
print(f'\nFlag-linked pool size: {int(mask.sum())}')


                  count      mean        std    min      25%      50%      75%        max
impressions_90d  9759.0  9321.448  21977.205  500.0  1295.00  3017.00  8248.50  517715.00
ctr              9759.0     0.185      0.134    0.0     0.08     0.17     0.28       0.49
avg_position     9759.0     9.483      4.564    0.2     5.90     8.40    12.70      20.00
engagement_rate  9759.0     3.077      7.147    0.0     0.00     0.00     3.57     100.00
scroll_rate      9756.0    10.516     15.997    0.0     0.00     4.88    13.33     200.00

Flag-linked pool size: 9759


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit suggests the queue should be built around observed opportunity, not raw traffic alone. High impressions and good visibility create a real opportunity set; then CTR and engagement tell us where the page is underperforming relative to that demand. That's a practical decision-support signal for editors and not a claim that Page 1 results are 'bad' or that changing titles always works.

In [4]:
print('Observed takeaways:')
print('- heavy-tailed demand means a few pages matter more than the average')
print('- CTR and engagement gap are real, measurable review opportunities')
print('- rule-based thresholds are useful, but a ranked queue is still better than a single if-statement')


Observed takeaways:
- heavy-tailed demand means a few pages matter more than the average
- CTR and engagement gap are real, measurable review opportunities
- rule-based thresholds are useful, but a ranked queue is still better than a single if-statement


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.